# Version 1
This is the most basic form that pv-finder can realistically take. While this 
does not implement manual layer creation (which would entirely use nn.Sequential 
and therefore have better performance), this was foregone with the knowledge 
that future versions of pv-finder will need more sophisticated logic

Changes:
- select_gpu() -> select_devices() which allows for any number of GPUs to be used
- Trying to put everything needed in just this notebook, will split up elsewhere into utilities once things work.

TODO:
- Remove config file bloat, only the essentials

## Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os
import time

In [2]:
# Set working directory to pv-finder root
os.chdir("/data/home/michael24peters/pv-finder")

## Select device

In [3]:
def select_devices(selection=None):
    """
    Select GPUs for use.

    Args:
        selection: GPU indices to use. Can be a single index (e.g., "0"), a
        comma-separated list of indices (e.g., "0,1"), or None to use all
        available GPUs. -1 to select CPU.
    """

    # All GPUs selection condition
    if selection is None:
        print("No GPU selection provided. Using all available GPUs.")
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Invalid selection condition
    elif not all(idx.strip().isdigit() for idx in selection.split(",")):
        raise ValueError("GPU selection must be a string of comma-separated indices or None.")
    # CPU selection condition
    elif selection == "-1": return torch.device("cpu")

    # GPU selection(s) condition
    # Set OS environment vars so that you don't need to specify device later.
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = selection

    # Check if cuda is available. If not, warn user.
    if not torch.cuda.is_available():
        print("Warning: GPU selected but CUDA not available. Using CPU.")
    else: 
        print(f"Using GPUs: {selection}")

    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    return device

In [4]:
# Display list of GPUs
!nvidia-smi

Thu Feb 12 04:20:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:18:00.0 Off |                  N/A |
| 29%   29C    P8             21W /  250W |      27MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

|    0   N/A  N/A            5878      G   /usr/libexec/Xorg                        16MiB |
|    0   N/A  N/A            5981      G   /usr/bin/gnome-shell                      5MiB |
+-----------------------------------------------------------------------------------------+


In [ ]:
from utils.utilities import select_gpu
device = select_gpu(2)

In [ ]:
# Device configuration
device = select_devices("2")

Using device: cpu


## Import configuration file

In [6]:
import yaml

CONFIG_FILE = '/data/home/michael24peters/pv-finder/configs/configuration.yaml'

with open(CONFIG_FILE, 'r') as file:
    configs = yaml.load(file, Loader=yaml.FullLoader)

EXPERIMENT = configs['Experiment']

global_configs = configs['global_configs'][EXPERIMENT]

## Data prepping

In [7]:
# Define data paths
# TODO: Create own paths and change in config file
# TODO: change config file name, i.e., saying "output..." for input file is
# confusing
data_root = configs['data_processing_configs']['output_files_path']

# TODO: get data
HDF_files = [data_root + '/' + 'pv_HLT1CPU_MinBiasMagUp_1__validPVs_5tracks__Intervals_100bins.hdf5',
             data_root + '/' + 'pv_HLT1CPU_MinBiasMagUp_2__validPVs_5tracks__Intervals_100bins.hdf5',
             data_root + '/' + 'pv_HLT1CPU_MinBiasMagUp_3__validPVs_5tracks__Intervals_100bins.hdf5',
             data_root + '/' + 'pv_HLT1CPU_MinBiasMagUp_4__validPVs_5tracks__Intervals_100bins.hdf5',
]

# TODO: remove get*TagName from utilities since mlflow is being used

### Convert ROOT files
In the beginning, we have a bunch of root files full of track information and poca KDE values, but it's not very optimized for machine learning. We want easy to slice, multidimensional arrays like numpy that for pytorch Dataset to use. This means we need to prepare this data.

Since there is a lot of data to process, we'll use hdf5 format, which gives us chunking, compression, etc. While we could use something like .npy, .pt, or other more familiar file types, hdf5 is good for our use case.

### Create datasets

Unfortunately, our data is not something that can simply be loaded in using standard pytorch tools alone.

In [8]:
# Get helper function
from utils.utilities import collectData_fromHDF

# Define splits [training, validation, test]
configs["training_configs"]["train_split"] = [0.8, 0.2, 0.0]

# Create datasets (if applicable)
train_loader, val_loader, _ = collectData_fromHDF(configs=configs,
                                                  HDF_files=HDF_files,
                                                  split=True,
                                                  verbose=False)
# TODO: make sure shuffle=True for train_loader, False for val_loader (and
# test_loader)

OSError: libc10_cuda.so: cannot open shared object file: No such file or directory

In [ ]:
## Inspect data
# TODO: Get sample and label from 0th index
# TODO: Print shapes and types
# TODO: Print labels and PV locations

In [ ]:
# TODO: Visualize sample and corresponding label

## Create the model
This will be the simplest form of the model possible, no bells and whistles, 
completely hardcoded.

In [ ]:
class SimpleCNN(nn.Module):
    """
    A simple Convolutional Neural Network model.

    The architecture consists of three convolutional blocks followed by two
    fully connected layers for classification.
    """
    def __init__(self):
        """
        Initializes the layers of the neural network.

        Args:
            num_classes: The number of output classes for the final layer.
        """
        # Call the constructor of the parent class (nn.Module)
        super(SimpleCNN, self).__init__()
        
        # Define repeating layers
        self.relu = nn.ReLU()
        # self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Define the first convolutional block
        self.conv1 = nn.Conv1d(in_channels=3, out_channels=32, kernel_size=3, 
                               padding=((3 - 1) // 2))  # same padding
        self.bn1 = nn.BatchNorm1d(32)
        
        # Define the second convolutional block
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, 
                               padding=((5 - 1) // 2))
        self.bn2 = nn.BatchNorm1d(64)
        
        # Define the third convolutional block
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=7, 
                               padding=((7 - 1) // 2))
        self.bn3 = nn.BatchNorm1d(64)

        # Define the fourth convolutional block
        self.conv4 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=5, 
                               padding=((5 - 1) // 2))
        self.bn4 = nn.BatchNorm1d(64)

        # Define the fifth convolutional block
        self.conv5 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=3, 
                               padding=((3 - 1) // 2))
        self.bn5 = nn.BatchNorm1d(64)
        
        # Define the layer to flatten the feature maps
        self.flatten = nn.Flatten()

        # Define the fully connected (dense) layers
        # Size is the same as input, but with 64 channels
        self.fc1 = nn.Linear(64 * 4000, 4000)
        # Start high since there are many parameters in fc1
        self.dropout = nn.Dropout(0.5)
        # Output size is the same as input size (4000) to match the labels
        self.fc2 = nn.Linear(4000, 4000)


    def forward(self, x):
        """
        Defines the forward pass of the model.

        Args:
            x: The input tensor of shape (batch_size, channels, length).

        Returns:
            The output tensor containing the logits for each class.
        """
        # Can't use MaxPool1d because we need the input and output size to be
        # the same (4000).

        # Pass input through the first convolutional block
        x = self.conv1(x)
        x = self.relu(x)
        x = self.bn1(x)

        # Pass feature maps through the second convolutional block
        x = self.conv2(x)
        x = self.relu(x)
        x = self.bn2(x)
        
        # Pass feature maps through the third convolutional block
        x = self.conv3(x)
        x = self.relu(x)
        x = self.bn3(x)

        # Pass feature maps through the fourth convolutional block
        x = self.conv4(x)
        x = self.relu(x)
        x = self.bn4(x)

        # Pass feature maps through the fifth convolutional block
        x = self.conv5(x)
        x = self.relu(x)
        x = self.bn5(x)

        # Flatten the output for the fully connected layers
        x = self.flatten(x)

        # Pass the flattened features through the fully connected layers
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        # Return the final output logits
        return x

In [ ]:
# Verify the SimpleCNN
print("--- Verifying SimpleCNN ---\n")

# Verify the structure of the model
# Instantiate the model with 15 output classes
verify_simple_cnn = SimpleCNN()
print("Model Structure:\n")
print(verify_simple_cnn)

# Verify the output shape after a forward pass
# Create a dummy input tensor (batch_size=64, channels=1, length=4000)
dummy_input = torch.randn(64, 1, 4000)
print(f"\nInput tensor shape:  {dummy_input.shape}")

# Pass the dummy tensor through the model
output = verify_simple_cnn(dummy_input)
print(f"Output tensor shape: {output.shape}")

In [ ]:
# Create model
model = SimpleCNN()

# Loss function
from models.Loss import Loss
loss_function = Loss(configs)

# Optimizer
# TODO: weight_decay=5e-4
# TODO: learning rate scheduler (e.g., StepLR, ReduceLROnPlateau, 
# CosineAnnealingLR)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# TODO: untangle mess model loops scattered all throughout pv-finder, in loss.py
# / alt_loss_A.py, training.py, utilities.py (progress bar, etc.), device
# allocation, epoch loop, train loop, validation loop
#
# I will sketch out the training loop here, but it will need to be refactored
# for pv-finder's specific needs and to ensure clean separation of concerns.

In [ ]:
# Define training loop for one epoch (one pass through the training data)
def train_epoch(model, loss_function, optimizer, train_loader, device):
    # Try to use multiple GPUs if available
    model= nn.DataParallel(model)
    # Move model to the appropriate device
    model.to(device)
    # Set model to training mode
    model.train()
    total_loss = 0
    n_correct_predictions, n_total_predictions = 0, 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to the appropriate device
        inputs, targets = inputs.to(device), targets.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = loss_function(outputs, targets)

        # Backpropagation to compute gradients
        loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate total loss
        total_loss += loss.data.item()

    return model, total_loss

In [ ]:
from utils.efficiency_utils import ValueSet, eval_efficiency
# Define one validation loop
def evaluate(model, val_loader, device):
    model.eval()  # Set model to evaluation mode
    n_correct_predictions, n_total_predictions = 0, 0
    # TODO: reinvent the wheel with ValueSet stuff (calculate metrics here)
    eff = ValueSet(0, 0, 0, 0)
    eff_params = configs['efficiency_config']

    with torch.no_grad():  # Disable gradient calculation for validation
        for inputs, labels in val_loader:
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            val_loss = loss_function(outputs, labels)
            total_loss += val_loss.data.item()

            # TODO: Simon does something weird for this section for intervals
            # where he switches to CPU, then redoes the outputs and loss, then
            # for intervals reshapes the data to be shaped as events, then 
            # calculates efficiency. Try it this way, see what happens, then
            # try it Simon's way and compare results.
            for i in range(len(inputs)):
                eff += eval_efficiency(labels[i], outputs[i], **eff_params)

            # Count correct predictions (see above)
            # predicted_indices = outputs.argmax(dim=1)
            # correct_predictions = predicted_indices.eq(labels)
            # n_correct_in_batch = correct_predictions.sum().item()
            # n_correct_predictions += n_correct_in_batch

            # Count total predictions (not using for now)
            # batch_size = len(labels)
            # n_total_predictions += batch_size
        
        # Calculate accuracy
        # accuracy = n_correct_predictions / n_total_predictions
        # print(f"Validation accuracy: {accuracy*100:.2f}%")
    # return accuracy
    return total_loss, eff

In [ ]:
# Train and evaluate the model for a number of epochs
def train_and_evaluate(model, loss_function, optimizer, train_loader, val_loader, device, n_epochs):
    for epoch in range(n_epochs):
        # Start epoch timer
        start_time = time.time()

        # Print current epoch
        print(f"Epoch {epoch + 1}/{n_epochs}")

        # Training step
        trained_model, total_train_loss = train_epoch(model, loss_function, optimizer, train_loader, device)
        
        # Validation step
        total_val_loss, val_efficiency = evaluate(model, val_loader, device)

        # Get epoch training and validation losses
        train_epoch = total_train_loss / len(train_loader)
        val_epoch = total_val_loss / len(val_loader)

        # Record total time
        epoch_duration = time.time() - start_time

        # Print metrics
        print(f"Epoch {epoch + 1} / {n_epochs}: Training loss: {train_epoch:.3f} | Validation loss: {val_epoch:.3f} | Efficiency: {val_efficiency} | Duration: {epoch_duration:.2f} seconds")
        
    return trained_model


In [ ]:
n_epochs = 100  # TODO: placeholder value
trained_model = train_and_evaluate(model=model, 
                                   loss_function=loss_function, 
                                   optimizer=optimizer, 
                                   train_loader=train_loader, 
                                   val_loader=val_loader, 
                                   device=device, 
                                   n_epochs=n_epochs)